## Statistical Analysis and Optimization: 
Analysing the data obtained and preparing it for Optimization

In [5]:
import numpy as np
import pandas as pd
import cvxpy as cp


In [9]:
df = pd.read_csv("Data.csv")
print(df)
returns = df.iloc[0, 1:]
returns

  Unnamed: 0      AAPL      MSFT      NVDA       JPM        GS       XOM  \
0     Return -0.328679 -0.878170 -0.064026 -0.442125 -0.367244  1.311107   
1        STD  0.240025  0.316209  0.337322  0.243065  0.338451  0.260398   

        JNJ      AMZN        PG       GLD  
0  0.618199 -0.341856  0.115720  0.268640  
1  0.158610  0.290512  0.207762  0.402045  


AAPL   -0.328679
MSFT    -0.87817
NVDA   -0.064026
JPM    -0.442125
GS     -0.367244
XOM     1.311107
JNJ     0.618199
AMZN   -0.341856
PG       0.11572
GLD      0.26864
Name: 0, dtype: object

The above given returns doesnt form a covariance matrix because, you need the time series data to establish more efficient covariance matrix, so we get the data from the time series histroy

In [4]:
# Load the full returns data
returns_df = pd.read_csv("returns_data.csv", index_col=0)  # Dates as index

# Compute the 10x10 covariance matrix
cov_matrix = returns_df.cov() * 252
print("Covariance Matrix (10x10):")
print(cov_matrix)

# Optional: Extract annualized means and stds
mean_returns = returns_df.mean() * 252
std_devs = returns_df.std() * (252 ** 0.5)
print("Mean Annual Returns:", mean_returns)
print("Annualized Std Devs:", std_devs)


Covariance Matrix (10x10):
          AAPL      MSFT      NVDA       JPM        GS       XOM       JNJ  \
AAPL  0.057640  0.003792  0.023985  0.019647  0.032419  0.001407  0.002535   
MSFT  0.003792  0.100920  0.029708  0.009711  0.016471 -0.017557 -0.007787   
NVDA  0.023985  0.029708  0.118240  0.019208  0.054922  0.001964  0.002023   
JPM   0.019647  0.009711  0.019208  0.059122  0.053599  0.001853 -0.012350   
GS    0.032419  0.016471  0.054922  0.053599  0.114723 -0.007834 -0.009963   
XOM   0.001407 -0.017557  0.001964  0.001853 -0.007834  0.067886  0.001579   
JNJ   0.002535 -0.007787  0.002023 -0.012350 -0.009963  0.001579  0.025746   
AMZN  0.013969  0.031704  0.013138  0.016124  0.015981 -0.016671 -0.013726   
PG    0.005194 -0.022162 -0.014055 -0.006527 -0.016676  0.008774  0.012643   
GLD  -0.007883  0.011335  0.027189  0.009343  0.016566  0.035384  0.007333   

          AMZN        PG       GLD  
AAPL  0.013969  0.005194 -0.007883  
MSFT  0.031704 -0.022162  0.011335  
NVD

What i have done above is by accessing the csv file, i have extracted the returns and standard deviations which are relevant for our optimization algorithm. 

Now we will be proceeding towards the very algorithm of Portfolio Optimization

Defining the decision variable: 

In [6]:
n = 10
w = cp.Variable(n)
w


Variable((10,), var1)

Now we need to define the objective function which as per Portfolio Optimization problem is:
Portfolio variance (risk):

$$ \min_w \quad w^T \Sigma w $$

In [8]:
obj = w.T @ cov_matrix @ w
obj_func = cp.Minimize(obj)
obj_func

Minimize(Expression(CONVEX, NONNEGATIVE, ()))


Now we need to set up the constraints:
1. Budget:
$$
w_1 + w_2 + w_3 + w_4 + ... + w_n = 1
$$

2. Target return:
$$
\mu^Tw \geq Expected \quad return
$$

3. No short selling:
$$
w_1, w_2 \geq 0
$$

$$

Obtaining the data on max return to set an expected return value:

In [26]:
maxi = max(returns)
# maxi
e_return = 0.15

Setting the constraints to cvxpy

In [25]:
mu = mean_returns.values
constraints = [sum(w) ==1, mu.T @ w >= e_return, w >= 0 ]

Defining the optimization problem and using the optimization algorithm from cvpxy

In [27]:
prob = cp.Problem(obj_func, constraints)
prob.solve()


np.float64(0.01010897238899978)

In [28]:
print(f"The Nature of the problem is ", prob.status)
print(f"The optimal value is", prob.value)
print(f"The weights that need to be assigned to the stocks are", w.value)
print(sum(w))

The Nature of the problem is  optimal
The optimal value is 0.01010897238899978
The weights that need to be assigned to the stocks are [-6.20836573e-24  5.92154525e-03  4.95479677e-24  4.30886074e-02
  4.36859355e-02  2.74232989e-01  4.94932657e-01  1.38138266e-01
 -1.64899265e-23  1.70996421e-23]
var1[0] + var1[1] + var1[2] + var1[3] + var1[4] + var1[5] + var1[6] + var1[7] + var1[8] + var1[9]


You may see there are some negative values, but you look properly these are actually raised to the power -23, which are nothing put numerical precision errors and can be safely treated as zeros. 